# Dev — sandbox Jev 0.8B

Quelques tests libres avec [guoan1/jev-qwen35-08b](https://huggingface.co/guoan1/jev-qwen35-08b).
Tout le calcul est **local**, sans clé API. Les poids sont téléchargés pendant
le setup, puis le notebook utilise le cache hors ligne.

Dans Jupyter Anaconda, choisir le kernel **Minecraft experiments - Jev 0.8B**.
Exécuter les cellules avec **Shift + Entrée**. Modifier les textes et les options,
puis relancer la cellule : le modèle reste chargé en mémoire.

Les exemples Minecraft sont écrits à la main : aucun bot ni serveur de jeu
n'est connecté à ce notebook. Les décisions et les scores sont à explorer,
pas des résultats validés pour le bot.


## 1. Vérifier l'environnement


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "jev_sandbox.py").is_file():
    ROOT = ROOT / "experiments"
if not (ROOT / "jev_sandbox.py").is_file():
    raise RuntimeError("Ouvrir ce notebook depuis le dossier experiments du projet Minecraft.")
expected_python = ROOT / ".venv" / "Scripts" / "python.exe"
if Path(sys.executable).resolve() != expected_python.resolve():
    raise RuntimeError("Sélectionner le kernel 'Minecraft experiments - Jev 0.8B' dans Jupyter, puis redémarrer le kernel.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from jev_sandbox import JevSandbox, environment
print(json.dumps(environment(), indent=2, ensure_ascii=False))


{
  "python": "3.14.0",
  "python_executable": "C:\\Users\\quent\\Desktop\\projects\\MinecraftLLM\\experiments\\.venv\\Scripts\\python.exe",
  "torch": "2.11.0+cu128",
  "cuda_available": true,
  "gpu": "NVIDIA GeForce RTX 5080",
  "gpu_free_gib": 14.63,
  "gpu_total_gib": 15.92
}


## 2. Charger le modèle une fois

`auto` choisit le GPU si au moins 4 Gio sont libres, sinon le CPU.
Utiliser `cpu` pour forcer le CPU, ou `cuda:0` pour tester le GPU.
Avant de changer de périphérique, redémarrer le kernel ou libérer `lab`.

La limite est de **768 tokens par entrée de candidat**, formatage compris.
Les longs états ou descriptions provoquent une erreur explicite.


In [12]:
DEVICE = "cpu"  # "auto", "cpu" ou "cuda:0"
if "lab" in globals():
    lab.close()
lab = JevSandbox(device=DEVICE)
print(f"Chargé sur {lab.device} en {lab.load_seconds:.1f} s ; limite : {lab.max_length} tokens.")


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 827.64it/s]


Chargé sur cpu en 1.4 s ; limite : 768 tokens.


## 3. Classifier un message — Choice

Modifier `text` et les descriptions des catégories. La sortie comprend le choix,
une probabilité pour chaque option et une confiance. `other` accueille les
messages hors périmètre. Commencer par un texte court.


In [13]:
text = "Collect 10 oak logs."
intent_question = {
    "intent": {
        "type": "choice",
        "instructions": "What is the player's main request to a Minecraft assistant?",
        "criteria": {
            "gather": "Collect or mine resources.",
            "build": "Build, place or craft something.",
            "follow": "Follow or come to the player.",
            "stop": "Stop or cancel the current task.",
            "other": "Chat, a question or another request.",
        },
    }
}
result = lab.decide({"player_message": text}, intent_question)
print(json.dumps(result, indent=2, ensure_ascii=False))


{
  "answers": {
    "intent": {
      "type": "choice",
      "choice": "gather",
      "probabilities": {
        "gather": 0.9313617328915111,
        "build": 0.023576740012922492,
        "follow": 0.006806182920908637,
        "stop": 0.0022736026749025974,
        "other": 0.03598174149975515
      },
      "confidence": 0.914202166114389
    }
  },
  "meta": {
    "model": "guoan1/jev-qwen35-08b",
    "device": "cpu",
    "elapsed_ms": 823.35,
    "input_tokens": 344,
    "temperature": 1.9190143561189226
  }
}


## 4. Poser une question oui/non — Noul

Modifier l'état et la question. `noul` est la probabilité que la réponse soit oui.
Comparer par exemple un inventaire de 4 puis de 12 blocs de bois.


In [11]:
state = {
    "goal": "Collect at least 10 oak logs (>= 10).",
    "inventory": {"oak_log": 9 },
}
questions = {
    "task_complete": {
        "type": "noul",
        "instructions": "Does the inventory satisfy the stated goal?",
    }
}
result = lab.decide(state, questions)
print(json.dumps(result, indent=2, ensure_ascii=False))


SyntaxError: invalid syntax (3836424142.py, line 3)

## 5. Tester un état Minecraft avec plusieurs questions

`choice` choisit une action ; `noul` évalue une condition ; `score` renvoie
la moyenne des niveaux d'une échelle décrite, ici de 0 à 2.
Les questions sont évaluées séparément pour réduire la mémoire utilisée.


In [ ]:
state = {
    "goal": "Continue gathering wood while staying alive.",
    "health": "3 hearts out of 10",
    "hunger": "Very hungry",
    "inventory": {"bread": 3, "oak_log": 4},
    "nearby": "A zombie is approaching. A safe house is five blocks away.",
}
questions = {
    "next_action": {
        "type": "choice",
        "instructions": "Which action should the assistant prioritize?",
        "criteria": {
            "gather": "Continue gathering wood.",
            "eat": "Eat bread to restore hunger.",
            "retreat": "Move into the safe house away from the zombie.",
            "wait": "Remain still and wait.",
        },
    },
    "danger": {"type": "noul", "instructions": "Is the player in immediate danger?"},
    "risk": {
        "type": "score",
        "instructions": "How risky is it to continue gathering wood right now?",
        "criteria": ["Low risk", "Moderate risk", "High risk"],
    },
}
result = lab.decide(state, questions)
print(json.dumps(result, indent=2, ensure_ascii=False))


## 6. Comparer des formulations et les temps (optionnel)

Six exemples anglais/français, avec des réponses attendues choisies manuellement.
Cela donne une première idée du comportement, pas une estimation générale de qualité.
Les temps couvrent le formatage, la tokenisation et le calcul, après le chargement
du modèle. Le premier appel peut être plus lent que les suivants.


In [ ]:
examples = [
    ("Collect 10 oak logs.", "gather"),
    ("Va chercher 10 blocs de bois de chêne.", "gather"),
    ("Build a small stone wall in front of me.", "build"),
    ("Viens me rejoindre.", "follow"),
    ("Stop your current task now.", "stop"),
    ("Quelle est la recette d'une pioche en fer ?", "other"),
]
observations = []
for message, expected in examples:
    response = lab.decide({"player_message": message}, intent_question)
    answer = response["answers"]["intent"]
    observation = {
        "text": message,
        "expected": expected,
        "predicted": answer["choice"],
        "correct": answer["choice"] == expected,
        "confidence": round(answer["confidence"], 3),
        "elapsed_ms": response["meta"]["elapsed_ms"],
        "device": response["meta"]["device"],
    }
    observations.append(observation)
    print(json.dumps(observation, ensure_ascii=False))
print(f"Sur ces seuls exemples : {sum(row['correct'] for row in observations)}/{len(observations)} réponses attendues retrouvées.")


## 7. Sauvegarder les résultats (optionnel)

Mettre `SAVE_RESULTS = True` pour sauvegarder les observations dans `runs/`,
ignoré par Git. Ne pas sauvegarder de données privées dans les sorties du notebook
si vous comptez ensuite le partager.


In [ ]:
SAVE_RESULTS = False
if SAVE_RESULTS:
    from datetime import datetime
    output_dir = ROOT / "runs"
    output_dir.mkdir(exist_ok=True)
    destination = output_dir / f"observations-{datetime.now():%Y%m%d-%H%M%S}.json"
    destination.write_text(json.dumps(observations, indent=2, ensure_ascii=False), encoding="utf-8")
    print(destination)


## 8. Libérer la mémoire (optionnel)

Mettre `RELEASE_MODEL = True` après les essais. Pour changer de CPU/GPU,
redémarrer le kernel, modifier `DEVICE`, puis relancer depuis le début.


In [ ]:
RELEASE_MODEL = False
if RELEASE_MODEL:
    lab.close()
    print("Modèle libéré.")
